# ✈️ Airline Passenger Satisfaction — Model Training

**Run this notebook in Google Colab.**

This notebook takes the raw survey dataset, cleans it, removes outliers, trains and
compares several classification models, and exports the winning model so it can be
plugged straight into the accompanying Flask web app.

**Steps covered:**
1. Load & inspect the data
2. Handle missing values
3. Remove outliers (IQR method) with before/after plots
4. Encode categorical features
5. Train/test split + scaling
6. Train and compare 5 models (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost)
7. Check for overfitting/underfitting with train vs. test vs. cross-validation scores
8. Evaluate the best model in depth (confusion matrix, ROC curve, feature importance)
9. Save `model.pkl` and `preprocessor.pkl` for the Flask app


## 0. Setup

If you're running this in Google Colab, upload `train.csv` first:
(`Files` panel on the left → upload button, or use the cell below)

In [ ]:
# Uncomment if running in Colab and you need to upload the dataset manually
# from google.colab import files
# uploaded = files.upload()  # select train.csv when prompted

!pip install xgboost -q


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

sns.set_style("whitegrid")
RANDOM_STATE = 42


## 1. Load & inspect the data

In [ ]:
df = pd.read_csv("train.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df['satisfaction'].value_counts(normalize=True)


## 2. Drop unhelpful columns

`Unnamed: 0` is just the row index and `id` is a passenger identifier — neither carries
any real signal about satisfaction, and keeping them risks the model latching onto
meaningless correlations (or, in the case of a monotonically increasing index, leaking
row order). Every other column is a genuine flight/service attribute, so nothing else
is dropped here.

In [ ]:
df = df.drop(columns=[c for c in ["Unnamed: 0", "id"] if c in df.columns])
df.shape


## 3. Handle missing values

Only `Arrival Delay in Minutes` has nulls (~0.3% of rows). Delay minutes are heavily
right-skewed (most flights are on time, a few are very late), so the **median** is a
safer fill than the mean, which outliers would drag upward.

In [ ]:
print(df['Arrival Delay in Minutes'].isnull().sum(), "missing values")
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(
    df['Arrival Delay in Minutes'].median()
)


## 4. Outlier removal (IQR method)

The 1–5 satisfaction rating columns are bounded and don't need outlier treatment.
`Flight Distance`, `Departure Delay in Minutes`, and `Arrival Delay in Minutes` are the
columns with genuine long-tail outliers (e.g. a handful of flights delayed by hours),
so we apply the interquartile-range rule to those three only.

In [ ]:
outlier_cols = ["Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df[col], ax=ax, color="#E8A33D")
    ax.set_title(f"{col} (before)")
plt.tight_layout()
plt.show()


In [ ]:
before = len(df)
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print(f"Removed {before - len(df)} rows ({(before - len(df)) / before:.1%})")
print("Shape after outlier removal:", df.shape)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df[col], ax=ax, color="#4F9D69")
    ax.set_title(f"{col} (after)")
plt.tight_layout()
plt.show()


## 5. Encode categorical features

`Gender`, `Customer Type`, `Type of Travel`, and `Class` are label-encoded so tree
models and linear models alike can consume them. The target (`satisfaction`) is
encoded too: `0 = neutral or dissatisfied`, `1 = satisfied`.

In [ ]:
categorical_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
target_col = "satisfaction"

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(col, "->", list(le.classes_))

target_encoder = LabelEncoder()
df[target_col] = target_encoder.fit_transform(df[target_col])
print(target_col, "->", list(target_encoder.classes_))


## 6. Train/test split + feature scaling

In [ ]:
numeric_cols = [c for c in df.columns if c not in categorical_cols + [target_col]]

X = df.drop(columns=[target_col])
y = df[target_col]
feature_order = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 7. Train and compare multiple models

We train five different classifiers and compare **train accuracy**, **test accuracy**,
and **5-fold cross-validation accuracy** side by side. This three-way comparison is
what lets us catch both:
- **Overfitting** — high train accuracy but a much lower test/CV accuracy (big gap)
- **Underfitting** — low accuracy across the board, even on training data

Logistic Regression uses the scaled features (it's distance/gradient based); the
tree-based models use the raw features (trees don't need scaling and it doesn't help
them).

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=14, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in models.items():
    use_scaled = name == "Logistic Regression"
    Xtr = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled if use_scaled else X_test

    model.fit(Xtr, y_train)

    train_pred = model.predict(Xtr)
    test_pred = model.predict(Xte)
    test_proba = model.predict_proba(Xte)[:, 1]

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring="accuracy", n_jobs=-1)

    results.append({
        "model": name,
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "cv_mean_accuracy": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "overfit_gap (train - test)": train_acc - test_acc,
        "precision": precision_score(y_test, test_pred),
        "recall": recall_score(y_test, test_pred),
        "f1": f1_score(y_test, test_pred),
        "roc_auc": roc_auc_score(y_test, test_proba),
    })

results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
results_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(results_df))
width = 0.25
ax.bar(x - width, results_df["train_accuracy"], width, label="Train", color="#E8A33D")
ax.bar(x, results_df["test_accuracy"], width, label="Test", color="#4F9D69")
ax.bar(x + width, results_df["cv_mean_accuracy"], width, label="CV mean", color="#C1440E")
ax.set_xticks(x)
ax.set_xticklabels(results_df["model"], rotation=20, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Train vs Test vs Cross-Validation Accuracy")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Select the best model

**Selection rule:** among models whose train/test accuracy gap is under 3% (this
rules out models that are clearly overfitting), pick the one with the highest test
accuracy. A low cross-validation standard deviation confirms the score is stable
across folds rather than lucky on one split.

In [ ]:
candidates = results_df[results_df["overfit_gap (train - test)"] < 0.03]
best_name = candidates.sort_values("test_accuracy", ascending=False).iloc[0]["model"]
best_model = models[best_name]
print("Best model selected:", best_name)


## 9. Evaluate the best model in depth

In [ ]:
use_scaled = best_name == "Logistic Regression"
Xte_final = X_test_scaled if use_scaled else X_test
final_pred = best_model.predict(Xte_final)
final_proba = best_model.predict_proba(Xte_final)[:, 1]

print(classification_report(y_test, final_pred, target_names=target_encoder.classes_))


In [ ]:
cm = confusion_matrix(y_test, final_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=target_encoder.classes_, yticklabels=target_encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, final_proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, color="#C1440E", label=f"ROC-AUC = {roc_auc_score(y_test, final_proba):.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve — {best_name}")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=feature_order).sort_values(ascending=False)
    plt.figure(figsize=(8, 6))
    sns.barplot(x=importances.head(12).values, y=importances.head(12).index, color="#4F9D69")
    plt.title(f"Top Feature Importances — {best_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
    importances.head(12)


## 10. Save the model for the Flask app

Two files are exported:
- `model.pkl` — the trained classifier
- `preprocessor.pkl` — everything needed to turn a raw form submission into a
  model-ready feature vector (label encoders, scaler, feature order, target
  encoder)

Copy both files into the Flask app's `model/` folder.

In [ ]:
joblib.dump(best_model, "model.pkl")
joblib.dump({
    "scaler": scaler,
    "encoders": encoders,
    "target_encoder": target_encoder,
    "feature_order": feature_order,
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "uses_scaling": use_scaled,
    "model_name": best_name,
}, "preprocessor.pkl")

print("Saved model.pkl and preprocessor.pkl")


In [ ]:
# If running in Colab, download the artifacts to your computer:
# from google.colab import files
# files.download("model.pkl")
# files.download("preprocessor.pkl")


## Why this model was chosen

The comparison table in step 7 is the actual justification — re-run this notebook
and read the printed numbers, since the exact figures depend on the random split.
In general, on this dataset the pattern looks like this:

- **Logistic Regression** underfits: it can only draw a straight decision boundary
  through the data, so both its train and test accuracy sit well below the tree-based
  models — it's too simple for the non-linear ways these features interact.
- **Decision Tree** (depth-limited) is a reasonable baseline but leaves accuracy on
  the table compared to ensembles, since a single tree can't average away noise.
- **Random Forest** and **Gradient Boosting** both perform strongly with small
  overfit gaps, because bagging/boosting many shallow trees smooths out the
  overfitting a single deep tree would show.
- **XGBoost** typically edges out the others on test accuracy and ROC-AUC while
  keeping a small train/test gap, thanks to its built-in regularization
  (`subsample`, `colsample_bytree`, shrinkage via `learning_rate`) — which is why it's
  usually the one this pipeline selects.

The selection logic itself (require overfit gap < 3%, then maximize test accuracy)
is what keeps the choice honest: a model can't win just by memorizing the training
set, it has to actually generalize.
